In [5]:
### Installing necessaary packages
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm

# Setup device (GPU if available, else CPU)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [2]:
### Mounting Drive to read the label file and image dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
train_path = '/content/drive/MyDrive/Case Studies/1. Toxic Comment Classification/train.csv'
test_path = '/content/drive/MyDrive/Case Studies/1. Toxic Comment Classification/test.csv'
test_lbl_path = '/content/drive/MyDrive/Case Studies/1. Toxic Comment Classification/test_labels.csv'

In [18]:
### Importing datasets
import pandas as pd
df = pd.read_csv(train_path)
df_test = pd.read_csv(test_path,encoding='latin')
df_labels = pd.read_csv(test_lbl_path)

In [21]:
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [22]:
df_test.head()

,id,comment_text
0,0001ea8717f6de06,Thank you for understanding. I think very high...
1,000247e83dcc1211,:Dear god this site is horrible.
2,0002f87b16116a7f,"""::: Somebody will invariably try to add Relig..."
3,0003e1cccfd5a40a,""" \n\n It says it right there that it IS a typ..."
4,00059ace3e3e9a53,""" \n\n == Before adding a new product to the l..."


In [8]:
df.columns

Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate'],
      dtype='object')

In [10]:
labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# Sum across the columns for each row
sumrow = df[labels].sum(axis=1)

# Count where sum is 0
clean_row = (sumrow == 0).sum()

# Count where sum is > 0
toxic_row = (sumrow > 0).sum()

print("Clean rows:", clean_row)
print("Toxic rows:", toxic_row)

Clean rows: 143346
Toxic rows: 16225


In [11]:
### Cleaning train dataset
import re

def fast_clean(text):
    # 1. Remove URLs first (so BS doesn't get confused)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 2. Remove HTML tags using Regex (much faster than BeautifulSoup)
    text = re.sub(r'<.*?>', '', text)

    # 3. Remove non-ASCII characters
    text = re.sub(r'[^\x00-\x7f]', r'', text)

    # 4. Cleanup whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply this to your dataframe
df['comment_text_cleaned'] = df['comment_text'].apply(fast_clean)

In [13]:
### Tokenizer Class
class ToxicityDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.tokenizer = tokenizer
        self.text = df['comment_text'].tolist()
        self.labels = df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].values
        self.max_len = max_len

    def __len__(self):
        return len(self.text)

    def __getitem__(self, index):
        text = str(self.text[index])

    # Tokenizer usage
        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
            )
        return {
          'input_ids': inputs['input_ids'].squeeze(0),
          'attention_mask': inputs['attention_mask'].squeeze(0),
          'labels': torch.tensor(self.labels[index], dtype=torch.float)
           }

In [14]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=6
)
model.to(device)

from torch.utils.data import random_split

dataset = ToxicityDataset(df, tokenizer)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
from torch.optim import AdamW

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3

# Mixed precision scaler
scaler = GradScaler()

for epoch in range(epochs):
    model.train()
    train_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['labels'].to(device)

        # Mixed precision forward
        with autocast():
            outputs = model(input_ids, attention_mask=attention_mask, labels=targets)
            loss = outputs.loss

        # Backprop with scaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        loop.set_postfix(loss=loss.item())

/tmp/ipykernel_3766/3192808081.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch 1/3 [Train]:   0%|          | 0/7979 [00:00<?, ?it/s]/tmp/ipykernel_3766/3192808081.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


In [16]:
train_loss /= len(train_loader)
# Validation
model.eval()
val_loss = 0
with torch.no_grad():
    loop_val = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]  ", leave=False)
    for batch in loop_val:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['labels'].to(device)

        with autocast():
            outputs = model(input_ids, attention_mask=attention_mask, labels=targets)
            val_loss += outputs.loss.item()
        loop_val.set_postfix(val_loss=outputs.loss.item())

val_loss /= len(val_loader)
print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")



Epoch 3/3 [Val]  :   0%|          | 0/1995 [00:00<?, ?it/s]/tmp/ipykernel_3766/2446899713.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
                                                                                         

Epoch 3/3 | Train Loss: 0.0273 | Val Loss: 0.0426


In [19]:
df_test_full = pd.merge(df_test, df_labels, on='id')
df_test_full = df_test_full[df_test_full['toxic'] != -1].reset_index(drop=True)

# Define test Dataset Class so that it matches the training setup
class ToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)
        }

target_columns = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

test_dataset = ToxicDataset(
    texts=df_test_full.comment_text.to_numpy(),
    labels=df_test_full[target_columns].to_numpy(),
    tokenizer=tokenizer # Use the same tokenizer as training
)

test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [29]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Switch model to evaluation mode
model.eval()

all_targets = []
all_preds = []
all_probs=[]

# No gradients needed during evaluation
with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        targets = batch['labels'].to(device)

        # Get model outputs
        outputs = model(input_ids, attention_mask=mask)
        logits = outputs.logits

        # Apply sigmoid since it's multi-label
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()  # threshold of 0.5

        all_targets.append(targets.cpu())
        all_preds.append(preds.cpu())
        all_probs.append(probs.cpu().numpy())

# Combine batches
#all_probs = all_probs.append(probs.cpu().numpy())
all_targets = torch.cat(all_targets, dim=0).numpy()
all_preds = torch.cat(all_preds, dim=0).numpy()

# Compute evaluation metrics
for i, col in enumerate(target_columns):
    auc = roc_auc_score(all_targets[:, i], all_preds[:, i])
    f1 = f1_score(all_targets[:, i], all_preds[:, i])
    print(f"{col}: AUC={auc:.4f}, F1={f1:.4f}")


toxic: AUC=0.9032, F1=0.6601
severe_toxic: AUC=0.7690, F1=0.4284
obscene: AUC=0.8789, F1=0.6992
threat: AUC=0.8773, F1=0.5316
insult: AUC=0.8617, F1=0.6865
identity_hate: AUC=0.8369, F1=0.6492


In [30]:
# Combine all batches into one array
import numpy as np
final_probs = np.vstack(all_probs)

In [32]:
final_probs

array([[2.26260643e-04, 1.13268281e-04, 1.60642740e-04, 5.71763085e-05,
        1.38959993e-04, 1.02300444e-04],
       [6.28782451e-01, 2.71620083e-04, 3.32272006e-03, 3.54114134e-04,
        4.72801039e-03, 5.81627188e-04],
       [4.47704375e-01, 9.39603196e-04, 6.10730909e-02, 7.34565139e-04,
        2.50044875e-02, 5.58448909e-03],
       ...,
       [9.91429389e-01, 1.58546213e-02, 1.62309125e-01, 5.16551035e-03,
        4.69121426e-01, 2.89728463e-01],
       [9.98058498e-01, 3.00660580e-01, 9.84401524e-01, 2.18828991e-02,
        9.76800442e-01, 8.28120887e-01],
       [3.03963170e-04, 8.67736817e-05, 1.39994299e-04, 3.70279267e-05,
        1.31951747e-04, 7.20384487e-05]], dtype=float32)

In [33]:
# Create the DataFrame
submission_df = pd.DataFrame(final_probs, columns=target_columns)
# Insert the ID column at the start
submission_df.insert(0, 'id', df_test_full['id'])
# Save to CSV
submission_df.to_csv('submission.csv', index=False)

print("Submission file created successfully!")

Submission file created successfully!
